# GPU-native neural CFR+ validation

This notebook validates target construction and exact resume first, then measures traversal speed and compares neural CFR+ with Deep CFR under equal GPU-training time.

In [ ]:
from pathlib import Path
import sys
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.algo.deep_cfr import DeepCFRTrainer
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.algo.neural_cfr_gpu import GPUDeepCFRTraverser
from liars_poker.algo.neural_cfr_plus_gpu import GPUDeepCFRPlusTraverser
from liars_poker.training.deep_cfr import deep_cfr_timed_loop
from liars_poker.training.deep_cfr_plus import deep_cfr_plus_timed_loop

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print('free / total GPU GiB:', round(free / 2**30, 2), '/', round(total / 2**30, 2))

## 1. Exact first-iteration target check

At iteration one, CFR+ targets must equal `relu(instantaneous regret)`. Resetting the Torch RNG makes Deep CFR and CFR+ receive identical deals and opponent samples.

In [ ]:
tiny_spec = GameSpec(
    ranks=2,
    suits=2,
    hand_size=1,
    claim_kinds=('RankHigh',),
    suit_symmetry=True,
)
common = dict(
    device='cpu',
    seed=3,
    batch_size=16,
    validation_fraction=0.0,
    traversal_backend='gpu_native',
    traversal_batch_size=32,
    device_replay=True,
    fused_optimizer=False,
)
deep = DeepCFRTrainer(
    tiny_spec,
    hidden_sizes=(8,),
    advantage_buffer_capacity=10_000,
    strategy_buffer_capacity=10_000,
    highest_regret_fallback=False,
    **common,
)
plus = DeepCFRPlusTrainer(
    tiny_spec,
    hidden_sizes=(8,),
    regret_buffer_capacity=10_000,
    strategy_buffer_capacity=10_000,
    **common,
)
deep.iteration = plus.iteration = 1

torch.manual_seed(99)
GPUDeepCFRTraverser(deep).run_traversals(0, 32)
torch.manual_seed(99)
GPUDeepCFRPlusTraverser(plus).run_traversals(0, 32)

n = deep.advantage_buffers[0].size
assert n == plus.regret_buffers[0].size
assert torch.equal(
    deep.advantage_buffers[0].features[:n],
    plus.regret_buffers[0].features[:n],
)
target_error = (
    torch.relu(deep.advantage_buffers[0].targets[:n])
    - plus.regret_buffers[0].targets[:n]
).abs().max().item()
assert target_error < 1e-6
print({'records': n, 'maximum target error': target_error})

## 2. Exact checkpoint continuation

Temporary per-update regret records are intentionally omitted from checkpoints: the next iteration clears them before use. Persistent strategy reservoirs and all RNG states must reproduce the next iteration exactly.

In [ ]:
resume_kwargs = dict(
    regret_hidden_sizes=(16, 16),
    strategy_hidden_sizes=(8, 8),
    device='cpu',
    seed=11,
    regret_buffer_capacity=10_000,
    strategy_buffer_capacity=10_000,
    batch_size=32,
    regret_train_steps=2,
    strategy_train_steps=2,
    validation_fraction=0.1,
    validation_buffer_capacity=1_000,
    traversal_backend='gpu_native',
    traversal_batch_size=32,
    device_replay=True,
    fused_optimizer=False,
)
uninterrupted = DeepCFRPlusTrainer(tiny_spec, **resume_kwargs)
for _ in range(2):
    uninterrupted.run_iteration(traversals_per_player=64)

with tempfile.TemporaryDirectory() as temporary_dir:
    checkpoint = Path(temporary_dir) / 'cfr_plus.pt'
    uninterrupted.save_checkpoint(checkpoint)
    uninterrupted.run_iteration(traversals_per_player=64)
    expected = [
        [parameter.detach().clone() for parameter in network.parameters()]
        for network in (*uninterrupted.regret_nets, *uninterrupted.strategy_nets)
    ]

    resumed = DeepCFRPlusTrainer.load_checkpoint(checkpoint, device='cpu')
    resumed.run_iteration(traversals_per_player=64)
    actual = [
        [parameter.detach() for parameter in network.parameters()]
        for network in (*resumed.regret_nets, *resumed.strategy_nets)
    ]

parameter_error = max(
    (left - right).abs().max().item()
    for left_network, right_network in zip(expected, actual)
    for left, right in zip(left_network, right_network)
)
assert parameter_error == 0.0
print({'maximum resumed parameter error': parameter_error})

## 3. Backend timing

The recursive row is a correctness reference. The GPU-native rows vary only traversal batch size.

In [ ]:
timing_spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'Trips'),
    suit_symmetry=True,
)
TIMING_TRAVERSALS = 256

def time_backend(backend, traversal_batch_size=None):
    backend_device = device if backend == 'gpu_native' else torch.device('cpu')
    trainer = DeepCFRPlusTrainer(
        timing_spec,
        regret_hidden_sizes=(256, 256),
        strategy_hidden_sizes=(128, 128),
        device=backend_device,
        seed=7,
        regret_buffer_capacity=200_000,
        strategy_buffer_capacity=200_000,
        batch_size=1024,
        regret_train_steps=10,
        strategy_train_steps=5,
        traversal_backend=backend,
        traversal_batch_size=traversal_batch_size or 256,
        device_replay=(backend == 'gpu_native'),
        fused_optimizer=(backend_device.type == 'cuda'),
    )
    record = trainer.run_iteration(traversals_per_player=TIMING_TRAVERSALS)
    records = sum(record['new_regret_records']) + sum(record['new_strategy_records'])
    traversal_s = record['timing']['traversal_s']
    return {
        'backend': backend,
        'device': str(backend_device),
        'traversal batch size': traversal_batch_size,
        'traversal s': traversal_s,
        'regret fit s': record['timing']['regret_training_s'],
        'strategy fit s': record['timing']['strategy_training_s'],
        'records': records,
        'records / traversal s': records / traversal_s,
    }

timing_rows = [time_backend('recursive')]
for traversal_batch_size in (64, 256, 1024):
    timing_rows.append(time_backend('gpu_native', traversal_batch_size))
backend_timing = pd.DataFrame(timing_rows)
backend_timing

## 4. Equal-time Deep CFR versus neural CFR+

Exact evaluation occurs only after each run and therefore does not consume either training budget. Increase `COMPARISON_SECONDS` after the short validation run succeeds.

In [ ]:
comparison_spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=3,
    claim_kinds=('RankHigh', 'Pair', 'Trips'),
    suit_symmetry=True,
)
COMPARISON_SECONDS = 5 * 60
TRAVERSALS_PER_PLAYER = 1024

shared = dict(
    strategy_hidden_sizes=(512, 512),
    device=device,
    seed=7,
    strategy_buffer_capacity=2_000_000,
    learning_rate=1e-3,
    batch_size=4096,
    strategy_train_steps=50,
    strategy_weighting='linear',
    validation_fraction=0.02,
    validation_buffer_capacity=20_000,
    traversal_backend='gpu_native',
    traversal_batch_size=1024,
    device_replay=True,
    fused_optimizer=(device.type == 'cuda'),
    amp_dtype=None,
    compile_models=False,
)
deep_kwargs = {
    **shared,
    'advantage_hidden_sizes': (2048, 2048),
    'advantage_buffer_capacity': 2_000_000,
    'advantage_train_steps': 100,
    'advantage_positive_weight': 0.5,
    'highest_regret_fallback': True,
    'alternating_updates': True,
}
plus_kwargs = {
    **shared,
    'regret_hidden_sizes': (2048, 2048),
    # CFR+ regret replay contains only the current player update, not all history.
    'regret_buffer_capacity': 500_000,
    'regret_train_steps': 100,
    'regret_positive_weight': 0.5,
}

deep_policy, deep_logs, deep_trainer = deep_cfr_timed_loop(
    comparison_spec,
    COMPARISON_SECONDS,
    trainer_kwargs=deep_kwargs,
    traversals_per_player=TRAVERSALS_PER_PLAYER,
    eval_every=0,
    final_eval=True,
    debug=True,
)
plus_policy, plus_logs, plus_trainer = deep_cfr_plus_timed_loop(
    comparison_spec,
    COMPARISON_SECONDS,
    trainer_kwargs=plus_kwargs,
    traversals_per_player=TRAVERSALS_PER_PLAYER,
    eval_every=0,
    final_eval=True,
    debug=True,
)

comparison_summary = pd.DataFrame([
    {
        'method': 'Deep CFR',
        'iterations': deep_trainer.iteration,
        'learned average exploitability': 2 * deep_logs['exploitability_series'][-1]['predicted_avg'] - 1,
        'current exploitability': 2 * deep_logs['exploitability_series'][-1]['current_predicted_avg'] - 1,
        'mean traversal s': np.mean([row['timing']['traversal_s'] for row in deep_logs['training_series']]),
        'mean main fit s': np.mean([row['timing']['advantage_training_s'] for row in deep_logs['training_series']]),
        'mean strategy fit s': np.mean([row['timing']['strategy_training_s'] for row in deep_logs['training_series']]),
    },
    {
        'method': 'neural CFR+',
        'iterations': plus_trainer.iteration,
        'learned average exploitability': 2 * plus_logs['exploitability_series'][-1]['predicted_avg'] - 1,
        'current exploitability': 2 * plus_logs['exploitability_series'][-1]['current_predicted_avg'] - 1,
        'mean traversal s': np.mean([row['timing']['traversal_s'] for row in plus_logs['training_series']]),
        'mean main fit s': np.mean([row['timing']['regret_training_s'] for row in plus_logs['training_series']]),
        'mean strategy fit s': np.mean([row['timing']['strategy_training_s'] for row in plus_logs['training_series']]),
    },
]).set_index('method')
comparison_summary